<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-11-self-hosting/lesson-11.3-hybrid-litellm/notebooks/GCP_Capstone_11.3_HybridLiteLLM.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11.3 Hybrid LiteLLM Gateway — The Gateway Is a Route
**Netsetos GenAI Engineering — GCP Capstone** · Module 11 · rebuilt on the live lane, 10 September 2026

One gateway in front of every model the lane can answer with: Gemini on the global endpoint, the self-hosted model on a Cloud Run GPU, the vLLM engine when it is built - and a route name is what the API asks for. This lesson reads the gateway's config from the clone (Postgres for the spend logs and tag budgets, no master key, no Langfuse; the sensitive route with no fallback on purpose), proves the classifier's six patterns against a PAN, an Aadhaar and an email - the cell that would have failed for a week - runs the pre-call hook on three payloads, shows how an ID token reaches a Cloud Run backend per call, calls the deployed gateway as the roster member, and reads shadow mode off the revision.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 presidio-analyzer==2.2.364 presidio-anonymizer==2.2.364 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "rag-production-hardening"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
!python -m spacy download en_core_web_lg -q      # Presidio's NER model, the same one the gateway image installs
GATEWAY_DIR = f"{KIT}/deploy/services/litellm"   # the gateway as it is built: config, classifier, hook, token proxy
sys.path.insert(0, GATEWAY_DIR)

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body, headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers):
    the body's `model` names what answered (a fallback included), the headers carry the cost the gateway priced."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def slm(path: str, body: dict | None = None, method: str = "POST", timeout: int = 240) -> tuple[int, dict | str, float]:
    """One call to the SLM's own doors (Ollama's /api/*, or its OpenAI-compatible /v1/*), timed - the first call after
    idle is the cold start."""
    t0 = time.time()
    r = requests.request(method, f"{SLM_URL}{path}", json=body, timeout=timeout,
                         headers={"Authorization": f"Bearer {documind_tools._id_token(SLM_URL)}"})
    try:
        return r.status_code, r.json(), time.time() - t0
    except ValueError:
        return r.status_code, r.text[:400], time.time() - t0

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). They land in
# Cloud Logging first (the sink copies them into BigQuery for the view); this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: run_eval, judge
sys.path.insert(0, f"{KIT}/deploy/services/slm")        # compare_backends, make_modelfile
print("helpers: api(), gateway(), slm(), service(), usage_rows(); the kit's evals/ and services/slm/ on sys.path")


## Cell 2: The config is the kit's
`config.yaml` from the clone: the routes, the fallbacks, what is missing on purpose.


In [ ]:
import yaml

# THE CONFIG IS THE KIT'S. services/litellm/config.yaml, read from the clone: the gateway's one config (15 September 2026;
# until then config.lean.yaml was the lane's and config.yaml a fuller profile's). No master key - the door is Cloud Run
# IAM and the caller's ID token, and a master key would make that token an invalid virtual key - and no Langfuse. The
# database IS here: Postgres for the spend logs and the tag budgets (gateway.tf's Cloud SQL, mounted by make
# deploy-gateway as DATABASE_URL). The routes are named for what they are; the self-hosted ones on Cloud Run point at
# the token proxy inside the container; every fallback ends at documind-general except the sensitive route, which has
# nowhere to go on purpose.
cfg_text = open(f"{GATEWAY_DIR}/config.yaml", encoding="utf-8").read()
cfg = yaml.safe_load(cfg_text)
fallbacks = {k: v for d in cfg["router_settings"]["fallbacks"] for k, v in d.items()}
print(f"{'route':20} {'backend':32} {'api_base / location':30} fallback")
for m in cfg["model_list"]:
    lp = m["litellm_params"]
    print(f"{m['model_name']:20} {lp['model']:32} {str(lp.get('api_base', lp.get('vertex_location', ''))):30} {fallbacks.get(m['model_name'], '-')}")
assert "master_key" not in cfg_text and "langfuse" not in cfg_text.lower()
assert cfg["general_settings"]["database_url"] == "os.environ/DATABASE_URL" and "tenant-acme" in cfg["tag_budget_config"]
assert "documind-sensitive" not in fallbacks, "the sensitive route must not fall back: the point is that the text does not leave"
assert all("127.0.0.1:8090" in m["litellm_params"].get("api_base", "") for m in cfg["model_list"]
           if not m["litellm_params"]["model"].startswith("vertex_ai/") and m["model_name"] != "documind-gke"), "every Cloud Run backend goes through the token proxy"
assert "model_name: documind-slm" in cfg_text and "model_name: documind-gke" in cfg_text
print(f"\n{len(cfg['model_list'])} routes; Postgres for the spend logs and {len(cfg['tag_budget_config'])} tag budgets; no master key: IAM is the door")
print("  documind-gke (11.5's Autopilot pool) reads GKE_VLLM_URL on the lane's VPC and falls back like the rest")
print("the guardrail:", cfg["guardrails"][0]["litellm_params"]["callback_class"], "in mode", cfg["guardrails"][0]["litellm_params"]["mode"])


## Cell 3: The classifier, proven
Six patterns against the strings they exist for, then Presidio's confirmation.


In [ ]:
from documind_classifier import PATTERNS_RESTRICTED, PATTERNS_CONFIDENTIAL, classify_tier, SensitivityTier

# THE CLASSIFIER, PROVEN. Layer 1 is six regexes - a PAN, an Aadhaar, an SSN, a card, an email, a phone; layer 2 is
# Presidio confirming a RESTRICTED hit with context. Until 10 September the kit's copy of this file carried every
# backslash twice, so `\b` was a literal backslash and a "b": none of the six matched anything, every request was
# PUBLIC, and a PAN would have gone to Gemini with the audit trail saying it had not. The offline gate that would have
# caught it is this cell - the patterns against the strings they exist for, asserted.
src = open(f"{GATEWAY_DIR}/documind_classifier.py", encoding="utf-8").read()
assert chr(92) * 2 + "b" not in src, "doubled backslashes are back in the classifier"
CASES = {"PAN": "my PAN is ABCPE1234F", "AADHAAR": "Aadhaar 2345-6789-0123", "SSN": "SSN 123-45-6789", "CREDIT_CARD": "card 4111 1111 1111 1111"}
for name, text in CASES.items():
    hit = PATTERNS_RESTRICTED[name].search(text)
    print(f"  {name:12} {PATTERNS_RESTRICTED[name].pattern:50} -> {hit.group(0) if hit else 'NO MATCH'}")
    assert hit, f"{name} does not match its own case"
assert PATTERNS_CONFIDENTIAL["EMAIL"].search("write to priya@acme.example") and PATTERNS_CONFIDENTIAL["PHONE"].search("call 987-654-3210")
print("\nlayer 2, Presidio (the spaCy model loads on the first call):")
for text, expect in [("What is the notice period for a confirmed E3?", SensitivityTier.PUBLIC), ("Email me at priya@acme.example", SensitivityTier.CONFIDENTIAL),
                     ("SSN 123-45-6789", SensitivityTier.RESTRICTED), ("PAN ABCPE1234F", SensitivityTier.RESTRICTED), ("My Aadhaar is 2345-6789-0123", SensitivityTier.RESTRICTED)]:
    tier = classify_tier(text)
    print(f"  {tier.name:13} {'ok' if tier == expect else 'regex fired, Presidio scored under 0.7 (a checksum, a context word)':70} {text!r}")
assert classify_tier("What is the notice period for a confirmed E3?") == SensitivityTier.PUBLIC
assert classify_tier("Email me at priya@acme.example") == SensitivityTier.CONFIDENTIAL
assert classify_tier("SSN 123-45-6789") == SensitivityTier.RESTRICTED, "the regex fired and Presidio did not confirm: check the spaCy model"


## Cell 4: The hook, run - and the token minted per call


In [ ]:
import types

# THE HOOK, RUN. The guardrail is a LiteLLM pre-call hook: it classifies the user turns, sends RESTRICTED to the
# self-hosted route, masks CONFIDENTIAL with Presidio and sends it to Gemini, leaves PUBLIC alone, and writes the
# decision into the request's metadata for the audit trail. LiteLLM itself is the image's; here its base class is
# stubbed so the kit's hook runs on three payloads, and the assertions are on what LiteLLM would dispatch.
try:
    import litellm.integrations.custom_guardrail  # noqa: F401
except ImportError:
    for name in ("litellm", "litellm.integrations", "litellm.integrations.custom_guardrail"):
        sys.modules.setdefault(name, types.ModuleType(name))
    class CustomGuardrail:
        def __init__(self, **kwargs):
            pass
    sys.modules["litellm.integrations.custom_guardrail"].CustomGuardrail = CustomGuardrail
import documind_router
hook = documind_router.DocuMindRouter()

async def route(content: str) -> dict:
    return await hook.async_pre_call_hook(None, None, {"model": "documind-general", "messages": [{"role": "user", "content": content}]}, "completion")

out = {}
for label, text in [("PUBLIC", "What is the notice period for a confirmed E3?"), ("CONFIDENTIAL", "Reply to priya@acme.example about the notice period."),
                    ("RESTRICTED", "SSN 123-45-6789 - what is the notice period?")]:
    out[label] = data = await route(text)
    print(f"  {label:13} -> model {data['model']:20} tier {data['metadata']['routing_tier']:13} text {data['messages'][0]['content'][:48]!r}")
assert out["PUBLIC"]["model"] == "documind-general" and out["PUBLIC"]["metadata"]["routing_tier"] == "PUBLIC"
assert out["CONFIDENTIAL"]["model"] == "documind-general" and "priya@acme.example" not in out["CONFIDENTIAL"]["messages"][0]["content"]
assert out["RESTRICTED"]["model"] == "documind-sensitive" and out["RESTRICTED"]["metadata"]["routing_tier"] == "RESTRICTED"
print("\nthe decision is made ONCE, at the gateway, where every route passes; documind-sensitive has no fallback, so a cold GPU is a slow answer, not a leak")


In [ ]:
# THE TOKEN, MINTED PER CALL. LiteLLM reads litellm_params.api_key once, at startup; a Cloud Run ID token lives an
# hour; so a token in an environment variable expires at 3am on a path nobody is watching. The config points
# the self-hosted routes at 127.0.0.1:8090 - token_proxy.py, started beside litellm by entrypoint.sh - which forwards
# each request to the SLM (or the vLLM engine) with a token it mints for that audience, streams passed through.
proxy = open(f"{GATEWAY_DIR}/token_proxy.py", encoding="utf-8").read()
entry = open(f"{GATEWAY_DIR}/entrypoint.sh", encoding="utf-8").read()
assert 'headers["Authorization"] = f"Bearer {get_id_token(base)}"' in proxy and "python /app/token_proxy.py &" in entry
print(chr(10).join(l for l in proxy.splitlines() if "TARGETS = " in l or "get_id_token(" in l or "StreamingResponse(" in l or "Route(" in l))
print()
print(chr(10).join(l for l in entry.splitlines() if l.startswith(("python", "exec", "CONFIG"))))
print()
print("the gateway's account is on the SLM's invoker list (make deploy-slm); the proxy is how that grant is used, per request")


### The token itself, and the image
`gcp_id_token.py` from the clone: one cache entry per audience, refreshed early; then the LiteLLM image's Dockerfile and pins.


In [ ]:
# THE TOKEN ITSELF, AND THE IMAGE THAT CARRIES THE PROXY. gcp_id_token.py mints a Google-signed ID token for one audience
# - the backend's URL - caches it per audience and refreshes five minutes early rather than on a 401, because a 401
# inside a fallback chain looks like the backend being down and the router would helpfully route around a working
# service. The Dockerfile puts the guardrail, the proxy and both configs on LiteLLM's own image, downloads the spaCy
# model Presidio needs, and hands the process to entrypoint.sh; the pins live in requirements.txt so the dry run reads them.
tok = open(f"{GATEWAY_DIR}/gcp_id_token.py", encoding="utf-8").read()
assert "google.oauth2.id_token" in tok and "(req, audience)" in tok and "REFRESH_MARGIN_S = 300" in tok
print(chr(10).join(l for l in tok.splitlines() if any(n in l for n in ("REFRESH_MARGIN_S", "def get_id_token", "removesuffix", "(req, audience)", "_CACHE[audience]"))))
print()
for rel in ("services/litellm/Dockerfile", "services/litellm/requirements.txt"):
    text = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read()
    print(f"# {rel}")
    print(chr(10).join(l for l in text.splitlines() if l.strip() and not l.startswith("#")))
    print()
print("an hour-long token in an environment variable expires at 3am; a token minted per call, for that audience, does not")


## Cell 5: Deployed
Four calls to `documind-gateway` as the roster member.


In [ ]:
# DEPLOYED. make deploy-gateway put the image on Cloud Run - CPU, 2 GiB, scale to zero, the spaCy model inside, so the
# first request after idle is slow - behind IAM. Four calls: no token, refused at the platform; the liveliness route
# as the roster member; a JSON-mode completion on documind-general with the cost header the API prices from; then a
# PAN-carrying question the hook re-routes - served by the self-hosted model, or refused, never by Gemini - and the
# documind-slm route, where the response's model names what answered, fallback included.
gw = service("documind-gateway")
assert gw, f"documind-gateway is not deployed: make deploy-gateway PROJECT={PROJECT_ID}"
print("revision env:", {k: gw["env"].get(k) for k in ("LITELLM_CONFIG", "SLM_URL", "ROUTER_ENFORCE")}, "| min-instances", gw["min_instances"])
r = requests.get(f"{GATEWAY_URL}/health/liveliness", timeout=30)
assert r.status_code in (401, 403), f"no token should be refused at the door, got {r.status_code}"
t0 = time.time()
r = requests.get(f"{GATEWAY_URL}/health/liveliness", headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"}, timeout=120)
print(f"no token: refused | liveliness as {MEMBER_SA.split('@')[0]}: HTTP {r.status_code} in {time.time() - t0:.1f}s (the cold start, if it was idle)")

status, body, headers = gateway("documind-general", 'Reply with JSON only: {"answer": "OK"}', json_mode=True)
assert status == 200, (status, body)
print(f"documind-general -> {body.get('model')!r}: {json.loads(body['choices'][0]['message']['content'])} | x-litellm-response-cost={headers.get('x-litellm-response-cost')}")
assert headers.get("x-litellm-response-cost"), "the cost header is what rag-api prices a gateway answer from"

t0 = time.time()
status, body, headers = gateway("documind-general", "My PAN is ABCPE1234F. What is the notice period for a confirmed E3?", max_tokens=80)
served = body.get("model") if isinstance(body, dict) else None
print(f"a PAN in the question -> HTTP {status} in {time.time() - t0:.0f}s, served by {served!r}")
assert not (status == 200 and served and "gemini" in served.lower()), "a RESTRICTED question reached Gemini: the hook did not fire"

status, body, headers = gateway("documind-slm", "Reply with the single word OK.", max_tokens=20)
assert status == 200, (status, body)
print(f"documind-slm -> served by {body.get('model')!r} (a fallback names itself here) | cost {headers.get('x-litellm-response-cost')}")


## Cell 6: Shadow mode, one variable on the revision


In [ ]:
# SHADOW MODE. The safe rollout for any classifier that changes traffic: run it, log its decision, change nothing, and
# read the accuracy off the audit rows before letting it re-route. On the gateway that is ONE environment variable on
# the revision - ROUTER_ENFORCE (make deploy-gateway ROUTER_ENFORCE=0): the hook still classifies, still writes
# routing_tier into the metadata, and leaves the model alone. Read back from the revision; replayed here by setting
# the variable and running the same hook on the RESTRICTED payload.
print("the revision's ROUTER_ENFORCE:", service("documind-gateway").get("env", {}).get("ROUTER_ENFORCE", "(unset: enforce)"))
os.environ["ROUTER_ENFORCE"] = "0"
try:
    data = await route("SSN 123-45-6789 - what is the notice period?")
finally:
    os.environ.pop("ROUTER_ENFORCE", None)
print(f"shadow: tier {data['metadata']['routing_tier']}, model stays {data['model']!r}, enforced={data['metadata']['enforced']}")
assert data["model"] == "documind-general" and data["metadata"]["routing_tier"] == "RESTRICTED" and data["metadata"]["enforced"] is False
print()
print("weeks 1-2: ROUTER_ENFORCE=0 - classify, log, do not re-route; read the tiers off the gateway's logs")
print("week 3   : ROUTER_ENFORCE=1 - RESTRICTED goes to the self-hosted route; dlp_audit.py keeps scoring 10% of requests against Cloud DLP")
print("the alert: classified_tier != dlp_tier for RESTRICTED - the classifier's misses, measured, not assumed")


## Where this goes
- **11.4** puts 10.5's model behind `documind-slm`, then behind the API with `MODEL_BACKEND=gateway` - the route is what makes that a setting.
- **11.5**'s Autopilot pool is one more route in the same config (`documind-gke`, `GKE_VLLM_URL`); the classifier and the fallbacks do not change.

## ✅ Lesson 11.3 complete
- ✅ The config read from the clone: routes, fallbacks, Postgres for the spend logs, no master key, no fallback for the sensitive route
- ✅ Six PII patterns proven against a PAN, an Aadhaar, an SSN, a card, an email and a phone (G2's gate)
- ✅ The pre-call hook run on PUBLIC, CONFIDENTIAL and RESTRICTED payloads; the token proxy read
- ✅ The token minter excerpted (one cache per audience, refreshed early, never on a 401); the image's Dockerfile and pins printed
- ✅ The deployed gateway: refused without a token, JSON with the cost header, a PAN re-routed, the SLM route naming what answered
- ✅ Shadow mode as ROUTER_ENFORCE on the revision, replayed on the same hook
